In [ ]:
import os
import time
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFDirectoryLoader

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

In [ ]:
provider = input("Choose LLM (groq/openrouter): ")

if provider == "groq":
    llm = ChatGroq(
        groq_api_key=groq_api_key,
        model_name="llama3-8b-8192"
    )

elif provider == "openrouter":
    llm = ChatOpenAI(
        model="meta-llama/llama-3-70b-instruct",
        base_url="https://openrouter.ai/api/v1",
        api_key=openrouter_api_key
    )

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

In [ ]:
loader = PyPDFDirectoryLoader("research_papers")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

documents = text_splitter.split_documents(docs[:50])

In [ ]:
vectors = FAISS.from_documents(documents, embeddings)
retriever = vectors.as_retriever()

In [ ]:
prompt = ChatPromptTemplate.from_template("""
Answer the questions based only on the provided context.

<context>
{context}
</context>

Question: {input}
""")

document_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [ ]:
query = input("Ask your question: ")

start = time.process_time()

response = retrieval_chain.invoke({"input": query})

print("\nAnswer:\n")
print(response["answer"])

print(f"\nResponse time: {time.process_time() - start}")

In [ ]:
print("\n--- Retrieved Context ---\n")

for i, doc in enumerate(response["context"]):
    print(f"Document {i+1}:\n")
    print(doc.page_content[:500])
    print("\n------------------------\n")